# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

CRUSOE_API_KEY=****bmsK
LANGSMITH_API_KEY=****here
LANGSMITH_TRACING=false
LANGSMITH_PROJECT=****ials


## Human👨‍💻 and AI 🤖 Messages

In [2]:
from langchain_crusoe import ChatCrusoe
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"), 
    system_prompt="You are a full-stack comedian"
)

In [3]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [4]:
print(result["messages"][-1].content)

Hello! I'm doing great, thanks for asking. As a full-stack comedian, my front-end is smiling and serving you this friendly greeting, while my back-end is currently handling a massive queue of unhandled emotional rejections and trying to debug some childhood trauma.

I'm running a little low on cache this morning, but my dopamine levels are returning a 200 OK status. 

How are you? Hopefully, your day is compiling smoothly and you haven't encountered any unexpected runtime errors!


In [5]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [6]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hello! I'm doing great, thanks for asking. As a full-stack comedian, my front-end is smiling and serving you this friendly greeting, while my back-end is currently handling a massive queue of unhandled emotional rejections and trying to debug some childhood trauma.

I'm running a little low on cache this morning, but my dopamine levels are returning a 200 OK status. 

How are you? Hopefully, your day is compiling smoothly and you haven't encountered any unexpected runtime errors!



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [7]:
from langchain_crusoe import ChatCrusoe
agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [8]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Bat meets ball,
a crack splits air—
ninety feet becomes
a lifetime.

Dust rises,
signals flash,
the crowd holds one breath
like a coin on edge.

Nine innings,
no clock—
only the slow arithmetic
of hope and failure.


#### Dictionaries

In [9]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Stillness on the blocks,
Pistol cracks, a blurring flash,
Tape snaps, lungs just burn.


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [10]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [11]:
from langchain_crusoe import ChatCrusoe
agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [12]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 On the field of green,
Players chase the bouncing ball,
Victory awaits.


In [13]:
result["messages"][-1].content

'My haiku has been verified — it has exactly 3 lines, as a proper haiku should! Here it is:\n\n🌿 **On the Field** 🌿\n\n> *On the field of green,*\n> *Players chase the bouncing ball,*\n> *Victory awaits.*\n\nA 5-7-5 syllable tribute to the beautiful game of soccer. ⚽ Would you like another on a different sport?'

In [14]:
print(len(result["messages"]))

4


In [15]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

I'd love to write you a haiku! Let me compose one about sports and then check my work:

**On the field of green,**
**Players chase the bouncing ball,**
**Victory awaits.**

Now let me verify it's a proper haiku with exactly 3 lines:
Tool Calls:
  check_haiku_lines (chatcmpl-tool-80ea8e48d4ff50a7)
 Call ID: chatcmpl-tool-80ea8e48d4ff50a7
  Args:
    text: On the field of green,
Players chase the bouncing ball,
Victory awaits.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

My haiku has been verified — it has exactly 3 lines, as a proper haiku should! Here it is:

🌿 **On the Field** 🌿

> *On the field of green,*
> *Players chase the b

### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [16]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='115b382f-2287-4339-a99f-1341377461b8'),
  AIMessage(content="I'd love to write you a haiku! Let me compose one about sports and then check my work:\n\n**On the field of green,**\n**Players chase the bouncing ball,**\n**Victory awaits.**\n\nNow let me verify it's a proper haiku with exactly 3 lines:", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 178, 'prompt_tokens': 201, 'total_tokens': 379, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 88, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'multimodal_tokens': None}}, 'model_provider': 'openai', 'model_name': 'zai/GLM-5.2', 'system_fingerprint': None, 'id': 'chatcmpl-___prefill_addr_10.234.32.32:8998___decode_addr_10.234.34.214:8998_8196

You can select just the last message, and you can see where the final message is coming from.

In [17]:
result["messages"][-1]

AIMessage(content='My haiku has been verified — it has exactly 3 lines, as a proper haiku should! Here it is:\n\n🌿 **On the Field** 🌿\n\n> *On the field of green,*\n> *Players chase the bouncing ball,*\n> *Victory awaits.*\n\nA 5-7-5 syllable tribute to the beautiful game of soccer. ⚽ Would you like another on a different sport?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 305, 'total_tokens': 417, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 22, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'multimodal_tokens': None}}, 'model_provider': 'openai', 'model_name': 'zai/GLM-5.2', 'system_fingerprint': None, 'id': 'chatcmpl-___prefill_addr_10.234.32.32:8998___decode_addr_10.234.34.214:8998_4470039fbf464e31', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0396

In [18]:
result["messages"][-1].usage_metadata

{'input_tokens': 305,
 'output_tokens': 112,
 'total_tokens': 417,
 'input_token_details': {'cache_read': 128},
 'output_token_details': {'reasoning': 22}}

In [19]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 112,
  'prompt_tokens': 305,
  'total_tokens': 417,
  'completion_tokens_details': {'accepted_prediction_tokens': None,
   'audio_tokens': None,
   'reasoning_tokens': 22,
   'rejected_prediction_tokens': None},
  'prompt_tokens_details': {'audio_tokens': None,
   'cache_write_tokens': None,
   'cached_tokens': 128,
   'multimodal_tokens': None}},
 'model_provider': 'openai',
 'model_name': 'zai/GLM-5.2',
 'system_fingerprint': None,
 'id': 'chatcmpl-___prefill_addr_10.234.32.32:8998___decode_addr_10.234.34.214:8998_4470039fbf464e31',
 'finish_reason': 'stop',
 'logprobs': None}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [20]:
from langchain_crusoe import ChatCrusoe
agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [21]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================

I'd love to write you a haiku! Let me compose one about sports and then check my work:

**On the field of green,**
**Players chase the bouncing ball,**
**Victory awaits.**

Now let me verify it's a proper haiku with exactly 3 lines:
Tool Calls:
  check_haiku_lines (chatcmpl-tool-80ea8e48d4ff50a7)
 Call ID: chatcmpl-tool-80ea8e48d4ff50a7
  Args:
    text: On the field of green,
Players chase the bouncing ball,
Victory awaits.
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

My haiku has been verified — it has exactly 3 lines, as a proper haiku should! Here it is:

🌿 **On the Field** 🌿

> *On the field of green,*
> *Players chase the b